In [ ]:
from http.cookiejar import cut_port_re

import numpy as np
import pandas as pd
import re # 두우노큐식정?

# 내용 확인용

In [ ]:
# 제조업만
nodong = pd.read_csv('data/노동비용(제조업).csv')
law_nodong = pd.read_csv('data/법정노동비용(제조업).csv')
notlaw_nodong = pd.read_csv('data/법정외_복지비용(제조업).csv')

# 전업종
nodong_total = pd.read_csv('data/노동비용(전업종).csv')
law_nodong_total = pd.read_csv('data/법정노동비용(전업종).csv')
notlaw_nodong_total = pd.read_csv('data/법정외_복지비용(전업종).csv')

# 산재
accident = pd.read_csv('data/전체재해현황및분석(제조업).csv')
accident_total = pd.read_csv('data/전체재해현황및분석(전업종).csv')

# 산업 규모뵬 임금, 근로시간
payment_time = pd.read_csv('data/산업규모및임금별근로시간(제조업).csv')
payment_time_total = pd.read_csv('data/산업규모및임금별근로시간(전업종).csv')
payment_time_part1 = pd.read_csv('data/payment_time_part1.csv')
payment_time_part2 = pd.read_csv('data/payment_time_part2.csv')

# 손익게산서 관련
sonic = pd.read_csv('data/손익계산서.csv')
sonic_jp = pd.read_csv('data/손익지표.csv')
sonic_jejo = pd.read_csv('data/손익계산서(제조업).csv')
sonic_jp_jejo = pd.read_csv('data/손익지표(제조업).csv')

In [ ]:
print(f'{nodong.shape} | {law_nodong.shape} | {notlaw_nodong.shape}')
print(f'{nodong_total.shape} | {law_nodong_total.shape} | {notlaw_nodong_total.shape}')
print(f'{accident.shape} | {accident_total.shape} | {payment_time.shape} | {payment_time_total.shape}')
print(f'{sonic.shape} | {sonic_jp.shape} | {sonic_jejo.shape} | {sonic_jp_jejo.shape}')

## nodong

## 헤더 나가!!

In [ ]:
# 1. 0번 행(노동비용총액 등)을 새로운 컬럼 이름으로 지정
nodong.columns = nodong.iloc[0]
nodong_total.columns = nodong_total.iloc[0]

# 2. 이름으로 써먹은 0번 행은 이제 데이터에서 삭제
nodong = nodong.drop(0)
nodong_total = nodong_total.drop(0)

# 너도 나가!
nodong.drop('기업규모별', axis=1, inplace=True)
nodong_total.drop('기업규모별', axis=1, inplace=True)

In [ ]:
nodong

In [ ]:
nodong_total

## 헤더 뭐뭐 있나 확인하기
- 근데... 뒤에서 가공해서 인스턴스로 다 빠질거면 굳이 단위는 안 빼도 되지 않나...?

In [ ]:
# 우리 아직 하나 더 남았습니다.
# 년도 앞에 2020~2024를 붙여야되는데 이게 10개단위거든요?
# 그죠 반복문 돌려야죠. 어느세월에 손으로 다 쓰고 앉아있음?

current_cols = nodong.columns.tolist() # 리스트로 가져옴
current_cols

# 여기서 0번 빼고 1번부터 바꿀거예요.
current_year = current_cols[1:]
print(current_year, len(current_year)) # 아 10개씩 끊으면 되네?

## 괄호씨는 더 이상 우리와 함께 할 수 없습니다.
- 아 아쉽습니다... 근데 빼야됩니다.

In [ ]:
pattern = r' \(천원\)' # 두유노정규식
current_cols = [re.sub(pattern, '', col) for col in current_cols]

current_cols

## 연도 앞으로 빼기

In [ ]:
years = [2019, 2020, 2021, 2022, 2023, 2024]
base_names = ['노동비용총액', '직접노동비용(계)', '정액 및 초과급여', '상여금 및 성과금', '간접노동비용(계)',
              '퇴직급여 등의 비용', '법정노동비용', '법정외 복지비용', '채용관련비용(모집비)', '교육훈련비용']

final_cols = []
for year in years:
    for name in base_names:
        final_cols.append(f"{year}_{name}")

# 3. 맨 앞에 '기업규모별' 컬럼이 있다면 추가 (데이터프레임 구조에 맞게)
if len(nodong.columns) == len(final_cols) + 1:
    final_cols = ['산업분류'] + final_cols

# 4. 이름표 갈아끼우기 (행은 전혀 건드리지 않음)
nodong.columns = final_cols
nodong_total.columns = final_cols # 둘이 범위만 다르고 칼럼이 같아서 이게 가능한겁니다.

In [ ]:
nodong_total

## 사르르르르르르

In [ ]:
nodong_melted = nodong.melt(id_vars=['산업분류'], var_name='항목', value_name='비용')
nodong_total_melted = nodong_total.melt(id_vars=['산업분류'], var_name='항목', value_name='비용')

nodong_melted[['연도', '항목']] = nodong_melted['항목'].str.split('_', expand=True, n=1)
nodong_total_melted[['연도', '항목']] = nodong_total_melted['항목'].str.split('_', expand=True, n=1)

nodong_melted['비용'] = pd.to_numeric(nodong_melted['비용'], errors='coerce')
nodong_total_melted['비용'] = pd.to_numeric(nodong_total_melted['비용'], errors='coerce')

nodong_melted = nodong_melted[['산업분류', '연도', '항목', '비용']]
nodong_total_melted = nodong_total_melted[['산업분류', '연도', '항목', '비용']]

In [ ]:
nodong_melted

In [ ]:
nodong_total_melted

## 저 앞에 알파벳들 되게 거슬린다 그죠?

In [ ]:
nodong_total_list = nodong_total_melted['산업분류'].tolist()
nodong_total_list

In [ ]:
re_pattern = r'[A-Z]{1}.' # 규식정 출동
nodong_total_list = [re.sub(re_pattern, '', col) for col in nodong_total_list]

nodong_total_melted['산업분류'] = nodong_total_list

nodong_total_melted

## 저 숫자도 되게 거슬린다 그죠?
- (00~00) 이거요.

In [ ]:
no_braket = r'\([0-9]+~?[0-9]+\)'
nodong_total_list = [re.sub(no_braket, '', col) for col in nodong_total_list]

nodong_total_melted['산업분류'] = nodong_total_list

nodong_total_melted

## 돈단위 변경(천원->만원)

In [ ]:
nodong_melted['비용(만원)'] = nodong_melted['비용'] / 10
nodong_total_melted['비용(만원)'] = nodong_total_melted['비용'] / 10

In [ ]:
nodong_melted

## 저장_최종.csv

In [ ]:
nodong_melted.to_csv('data/nodong.csv', index=False) # 다음에는 후가공 필요없이 이걸로 하면 되지요.
nodong_total_melted.to_csv('data/nodong_total.csv', index=False)

# law_nodong

In [ ]:
law_nodong

## 헤더 나가! 

In [ ]:
# 1. 0번 행(노동비용총액 등)을 새로운 컬럼 이름으로 지정
law_nodong.columns = law_nodong.iloc[0]
law_nodong_total.columns = law_nodong_total.iloc[0]

# 2. 이름으로 써먹은 0번 행은 이제 데이터에서 삭제
law_nodong = law_nodong.drop(0)
law_nodong_total = law_nodong_total.drop(0)

# 너도 나가!
law_nodong.drop('기업규모별', axis=1, inplace=True)
law_nodong_total.drop('기업규모별', axis=1, inplace=True)

In [ ]:
law_nodong

In [ ]:
law_nodong_total

In [ ]:
current_cols = law_nodong_total.columns
current_cols

## 괄호 나가

In [ ]:
pattern = r' \(천원\)' # 두유노정규식
current_cols = [re.sub(pattern, '', col) for col in current_cols] # (천원은 다 빼주시고)
current_cols = [re.sub(r' \(%\)', '_(%)', col) for col in current_cols] # 퍼센트도 퇴근합니당

current_cols

## 연도 나와

In [ ]:
years = [2019, 2020, 2021, 2022, 2023, 2024]
base_names = ['법정노동비용(계)','건강보험료','산재보험료','국민연금','고용보험료','장애인고용부담금','재해보상비','구성비(계)','건강보험료_(%)','산재보험료_(%)','국민연금_(%)','고용보험료_(%)','장애인고용부담금_(%)','재해보상비_(%)']

final_cols = []
for year in years:
    for name in base_names:
        final_cols.append(f"{year}_{name}")

# 3. 맨 앞에 '기업규모별' 컬럼이 있다면 추가 (데이터프레임 구조에 맞게)
if len(law_nodong.columns) == len(final_cols) + 1:
    final_cols = ['산업분류'] + final_cols

# 4. 이름표 갈아끼우기 (행은 전혀 건드리지 않음)
law_nodong.columns = final_cols
law_nodong_total.columns = final_cols

In [ ]:
law_nodong_total

## 잠시만요 분리좀 하고 가실게요!
- 단위가 (천원)인 것과 (%)인 걸로 분리할겁니다.

In [ ]:
# 공통분모
union_column = ['산업분류']

# 그룹 1: 단위가 (천원)임 ('법정노동비용(계) (천원)', '건강보험료 (천원)', '산재보험료 (천원)', '국민연금 (천원)', '고용보험료 (천원)','장애인고용부담금 (천원)', '재해보상비 (천원)'
won_mnu_cols = [
    col for col in law_nodong.columns
    if any(x in col for x in ['법정노동비용(계)','건강보험료','산재보험료','국민연금','고용보험료','장애인고용부담금','재해보상비'])
    and '%' not in col  # 이 조건을 추가해서 (%) 항목을 걸러냅니다.
]
law_nodong_mnu_won = law_nodong_total.melt(id_vars=union_column, value_vars=won_mnu_cols, var_name='항목', value_name='비용')

# 그룹 2: % '구성비(계) (%)', '건강보험료 (%)', '산재보험료 (%)', '국민연금 (%)', '고용보험료 (%)', '장애인고용부담금 (%)', '재해보상비 (%)'
rate_mnu_cols = [col for col in law_nodong.columns if any(x in col for x in ['구성비(계)','건강보험료_(%)','산재보험료_(%)','국민연금_(%)','고용보험료_(%)','장애인고용부담금_(%)','재해보상비_(%)'])]
law_nodong_mnu_rate = law_nodong_total.melt(id_vars=union_column, value_vars=rate_mnu_cols, var_name='항목', value_name='수치')

# 연도와 지표명 깔끔하게 분리 (예: 2019_사업장수 (개소) -> 2019 / 사업장수)
for df in [law_nodong_mnu_won, law_nodong_mnu_rate]:
    df[['연도', '지표']] = df['항목'].str.split('_', expand=True, n=1)
    df['지표'] = df['지표'].str.replace(r' \(.*\)', '', regex=True) # 단위 제거
    df.drop(columns=['항목'], inplace=True)

In [ ]:
# 공통분모
union_column = ['산업분류']

# 그룹 1: 단위가 (천원)임 ('법정노동비용(계) (천원)', '건강보험료 (천원)', '산재보험료 (천원)', '국민연금 (천원)', '고용보험료 (천원)','장애인고용부담금 (천원)', '재해보상비 (천원)'
won_cols = [
    col for col in law_nodong.columns
    if any(x in col for x in ['법정노동비용(계)','건강보험료','산재보험료','국민연금','고용보험료','장애인고용부담금','재해보상비'])
    and '%' not in col  # 이 조건을 추가해서 (%) 항목을 걸러냅니다.
]
law_nodong_won = law_nodong_total.melt(id_vars=union_column, value_vars=won_cols, var_name='항목', value_name='비용')

# 그룹 2: % '구성비(계) (%)', '건강보험료 (%)', '산재보험료 (%)', '국민연금 (%)', '고용보험료 (%)', '장애인고용부담금 (%)', '재해보상비 (%)'
rate_cols = [col for col in law_nodong_total.columns if any(x in col for x in ['구성비(계)','건강보험료_(%)','산재보험료_(%)','국민연금_(%)','고용보험료_(%)','장애인고용부담금_(%)','재해보상비_(%)'])]
law_nodong_rate = law_nodong_total.melt(id_vars=union_column, value_vars=rate_cols, var_name='항목', value_name='수치')

# 연도와 지표명 깔끔하게 분리 (예: 2019_사업장수 (개소) -> 2019 / 사업장수)
for df in [law_nodong_won, law_nodong_rate]:
    df[['연도', '지표']] = df['항목'].str.split('_', expand=True, n=1)
    df['지표'] = df['지표'].str.replace(r' \(.*\)', '', regex=True) # 단위 제거
    df.drop(columns=['항목'], inplace=True)

In [ ]:
law_nodong_won = law_nodong_won[['산업분류', '연도', '지표' ,'비용']]
law_nodong_mnu_won = law_nodong_mnu_won[['산업분류', '연도', '지표' ,'비용']]
law_nodong_won['지표'].value_counts()

In [ ]:
law_nodong_rate = law_nodong_rate[['산업분류', '연도', '지표' ,'수치']]
law_nodong_mnu_rate = law_nodong_mnu_rate[['산업분류', '연도', '지표' ,'수치']]
law_nodong_rate['지표'].value_counts()

## 너도 알파벳 떼고 나가자

In [ ]:
law_nodong_won_list = law_nodong_won['산업분류'].tolist()
law_nodong_rate_list = law_nodong_rate['산업분류'].tolist()

In [ ]:
re_pattern = r'[A-Z]{1}.' # 규식정 출동

law_nodong_rate_list = [re.sub(re_pattern, '', col) for col in law_nodong_rate_list]
law_nodong_won_list = [re.sub(re_pattern, '', col) for col in law_nodong_won_list]

law_nodong_rate['산업분류'] = law_nodong_rate_list
law_nodong_won['산업분류'] = law_nodong_won_list

law_nodong_won

In [ ]:
no_braket = r'\([0-9]+~?[0-9]+\)'

# 뒤에 숫자도 나가자
law_nodong_rate_list = [re.sub(no_braket, '', col) for col in law_nodong_rate_list]
law_nodong_won_list = [re.sub(no_braket, '', col) for col in law_nodong_won_list]

law_nodong_rate['산업분류'] = law_nodong_rate_list
law_nodong_won['산업분류'] = law_nodong_won_list

law_nodong_rate

In [ ]:
law_nodong_rate['지표'] = law_nodong_rate['지표'].str.replace('_(%)', '', regex=False)
law_nodong_mnu_rate['지표'] = law_nodong_mnu_rate['지표'].str.replace('_(%)', '', regex=False)
law_nodong_rate

In [ ]:
law_nodong_mnu_rate_list = [re.sub(re_pattern, '', col) for col in law_nodong_rate_list]
law_nodong_mnu_won_list = [re.sub(re_pattern, '', col) for col in law_nodong_won_list]

law_nodong_mnu_rate_list = [re.sub(no_braket, '', col) for col in law_nodong_rate_list]
law_nodong_mnu_won_list = [re.sub(no_braket, '', col) for col in law_nodong_won_list]

law_nodong_mnu_rate['산업분류'] = law_nodong_rate_list
law_nodong_mnu_won['산업분류'] = law_nodong_won_list

law_nodong_mnu_rate

## 돈단위 변경 (천원->만원)

In [ ]:
# 아놔... 얼탱이없네... 이게 왜 숫자냐고...
law_nodong_won['비용'] = law_nodong_won['비용'].astype(float)
law_nodong_mnu_won['비용'] = law_nodong_mnu_won['비용'].astype(float)

In [ ]:
law_nodong_won['비용(만원)'] = law_nodong_won['비용'] / 10
law_nodong_mnu_won['비용(만원)'] = law_nodong_mnu_won['비용'] / 10

## 저장_최종.csv

In [ ]:
law_nodong_mnu_rate.to_csv('data/law_nodong_rate.csv', index=False)
law_nodong_mnu_won.to_csv('data/law_nodong_won.csv', index=False)
law_nodong_rate.to_csv('data/law_nodong_total_rate.csv', index=False)
law_nodong_won.to_csv('data/law_nodong_total_won.csv', index=False)

# notlaw_nodong
## 헤더 나갓! 

In [ ]:
# 1. 0번 행(노동비용총액 등)을 새로운 컬럼 이름으로 지정
notlaw_nodong.columns = notlaw_nodong.iloc[0]
notlaw_nodong_total.columns = notlaw_nodong_total.iloc[0]

# 2. 이름으로 써먹은 0번 행은 이제 데이터에서 삭제
notlaw_nodong= notlaw_nodong.drop(0)
notlaw_nodong_total = notlaw_nodong_total.drop(0)

# 너도 나가!
notlaw_nodong.drop('기업규모별', axis=1, inplace=True)
notlaw_nodong_total.drop('기업규모별', axis=1, inplace=True)

In [ ]:
notlaw_nodong

In [ ]:
notlaw_nodong_total

In [ ]:
current_cols = notlaw_nodong_total.columns

## 괄호 나갓!!

In [ ]:
pattern = r' \(천원\)' # 두유노정규식
current_cols = [re.sub(pattern, '', col) for col in current_cols] # (천원은 다 빼주시고)
current_cols = [re.sub(r' \(%\)', '', col) for col in current_cols] # 퍼센트도 퇴근합니당

current_cols

## 연도 분리 및 멜트

In [ ]:
years = [2019, 2020, 2021, 2022, 2023, 2024]
base_names = ['법정외 복지비용(계)','주거비용','건강·보건비용','식사비용','교통·통신지원비용','보육지원금','보험료지원금','자녀학비보조비용','휴양·문화·체육·오락비용','우리사주제도 지원금','사내근로복지기금 출연금','기타']

final_cols = []
for year in years:
    for name in base_names:
        final_cols.append(f"{year}_{name}")

# 3. 맨 앞에 '기업규모별' 컬럼이 있다면 추가 (데이터프레임 구조에 맞게)
if len(notlaw_nodong.columns) == len(final_cols) + 1:
    final_cols = ['산업분류'] + final_cols

# 4. 이름표 갈아끼우기 (행은 전혀 건드리지 않음)
notlaw_nodong.columns = final_cols
notlaw_nodong_total.columns = final_cols

In [ ]:
notlaw_nodong_melted = notlaw_nodong.melt(id_vars=['산업분류'], var_name='항목', value_name='비용')
notlaw_nodong_total_melted = notlaw_nodong_total.melt(id_vars=['산업분류'], var_name='항목', value_name='비용')

notlaw_nodong_melted[['연도', '항목']] = notlaw_nodong_melted['항목'].str.split('_', expand=True, n=1)
notlaw_nodong_total_melted[['연도', '항목']] = notlaw_nodong_total_melted['항목'].str.split('_', expand=True, n=1)

notlaw_nodong_melted['비용'] = pd.to_numeric(notlaw_nodong_melted['비용'], errors='coerce')
notlaw_nodong_total_melted['비용'] = pd.to_numeric(notlaw_nodong_total_melted['비용'], errors='coerce')

notlaw_nodong_melted = notlaw_nodong_melted[['산업분류', '연도', '항목', '비용']]
notlaw_nodong_total_melted = notlaw_nodong_total_melted[['산업분류', '연도', '항목', '비용']]

In [ ]:
notlaw_nodong_melted

In [ ]:
notlaw_nodong_total_melted

## 알파벳 떼고 가실게요!

In [ ]:
notlaw_nodong_total_list = notlaw_nodong_melted['산업분류'].tolist()
notlaw_nodong_total_list

In [ ]:
notlaw_nodong_total_list = [re.sub(re_pattern, '', col) for col in notlaw_nodong_total_list]

notlaw_nodong_melted['산업분류'] = notlaw_nodong_total_list

notlaw_nodong_melted

In [ ]:
# 뒤에 숫자도 나가자
notlaw_nodong_total_list = [re.sub(no_braket, '', col) for col in notlaw_nodong_total_list]

notlaw_nodong_melted['산업분류'] = notlaw_nodong_total_list

notlaw_nodong_melted

## 돈단위 변경 (천원->만원)

In [ ]:
notlaw_nodong_melted['비용(만원)'] = notlaw_nodong_melted['비용'] / 10
notlaw_nodong_total_melted['비용(만원)'] = notlaw_nodong_total_melted['비용'] / 10

## 저장_최종.csv

In [ ]:
notlaw_nodong_melted.to_csv('data/notlaw_nodong_total.csv', index=False)
notlaw_nodong_total_melted.to_csv('data/notlaw_nodong.csv', index=False)
# 아놔 이름을 바꿔서 저장했어요... ㅡㅡ

- 아놔 둘이 이름을 바꿔서 저장했네요.. 내용이 뭔가 이상하더라니... ㅡㅡ 

# 산업재해 관련 처리

## 내용 좀 보고 가겠습니다.

In [ ]:
accident

In [ ]:
accident_total

- 이것도 손 많이 가겠는데...?

## 이제는 헤더가 퇴근해야 할 시간

In [ ]:
# 1. 0번 행(노동비용총액 등)을 새로운 컬럼 이름으로 지정
accident.columns = accident.iloc[0]
accident_total.columns = accident_total.iloc[0]

# 2. 이름으로 써먹은 0번 행은 이제 데이터에서 삭제
accident = accident.drop(0)
accident_total = accident_total.drop(0)

# 너도 나가!
accident.drop('업종별 중분류(1)', axis=1, inplace=True) # 얘 중분류 날립니다 (어차피 제조업 하나임)

In [ ]:
accident

In [ ]:
accident_total

## melt하기 전 처리
1. 연도가 2019~2024라서 그거 따로 붙여줄거고요
2. 우리 저 칼럼 보시면 (1) 있죠? 그거 떼버릴겁니다.
### 괄호는 안녕~

In [ ]:
# (1), (2) 떼고
accident.columns = accident.columns.str.strip()
accident_total.columns = accident_total.columns.str.strip()

accident.rename(columns={'업종별 중분류(2)':'업종별 중분류'}, inplace=True)
accident_total.rename(columns={'업종별 중분류(1)':'업종별 중분류'}, inplace=True)

In [ ]:
accident

In [ ]:
accident_total.columns

### 연도별로 구별해줄거에요

In [ ]:
# 연도향을 첨가해보아요
years = [2019, 2020, 2021, 2022, 2023, 2024]
base_names = ['사업장수 (개소)', '근로자수 (명)', '재해자수 (명)', '사망자수 (명)', '재해율 (%)',
       '사망만인율 (‱)'] # total도 동일

final_cols = []
for year in years:
    for name in base_names:
        final_cols.append(f"{year}_{name}")

# 3. 맨 앞에 '기업규모별' 컬럼이 있다면 추가 (데이터프레임 구조에 맞게)
if len(accident.columns) == len(final_cols) + 1:
    final_cols = ['업종별 중분류'] + final_cols

# 4. 이름표 갈아끼우기 (행은 전혀 건드리지 않음)
accident.columns = final_cols
accident_total.columns = final_cols

In [ ]:
# 굿.
accident_total

In [ ]:
accident.columns

## 멜트로 재구성
- 이거 단위가 섞여있어서 하나를 둘로 찢을거예요. 하나는 사업장 수 대비 근로자수, 재해자수, 사망자수가 들어가고 다른 하나는 사업장 수 대비 재해율, 사망만인율(...퍼밀인가?)이 들어갈겁니다.

In [ ]:
# union_column = ['업종별 중분류']
# 그룹 1: 규모 및 수량 (사업장수, 근로자수, 재해자수, 사망자수)
qty_cols = [col for col in accident.columns if any(x in col for x in ['사업장수', '근로자수', '재해자수', '사망자수'])]
accident_quantity = accident.melt(id_vars=union_column, value_vars=qty_cols, var_name='항목', value_name='수치')

# 그룹 2: 비율 및 위험도 (재해율, 사망만인율)
rate_cols = [col for col in accident.columns if any(x in col for x in ['재해율', '사망만인율'])]
accident_rate = accident.melt(id_vars=union_column, value_vars=rate_cols, var_name='항목', value_name='수치')

# 연도와 지표명 깔끔하게 분리 (예: 2019_사업장수 (개소) -> 2019 / 사업장수)
for df in [accident_quantity, accident_rate]:
    df[['연도', '지표']] = df['항목'].str.split('_', expand=True, n=1)
    df['지표'] = df['지표'].str.replace(r' \(.*\)', '', regex=True) # 단위 제거
    df.drop(columns=['항목'], inplace=True)

In [ ]:
union_column = ['업종별 중분류']

# 그룹 1: 규모 및 수량 (사업장수, 근로자수, 재해자수, 사망자수)
qty_cols = [col for col in accident.columns if any(x in col for x in ['사업장수', '근로자수', '재해자수', '사망자수'])]
accident_quantity = accident.melt(id_vars=union_column, value_vars=qty_cols, var_name='항목', value_name='수치')

# 그룹 2: 비율 및 위험도 (재해율, 사망만인율)
rate_cols = [col for col in accident.columns if any(x in col for x in ['재해율', '사망만인율'])]
accident_rate = accident.melt(id_vars=union_column, value_vars=rate_cols, var_name='항목', value_name='수치')

# 연도와 지표명 깔끔하게 분리 (예: 2019_사업장수 (개소) -> 2019 / 사업장수)
for df in [accident_quantity, accident_rate]:
    df[['연도', '지표']] = df['항목'].str.split('_', expand=True, n=1)
    df['지표'] = df['지표'].str.replace(r' \(.*\)', '', regex=True) # 단위 제거
    df.drop(columns=['항목'], inplace=True)

In [ ]:
union_column = ['업종별 중분류']

# 그룹 1: 규모 및 수량 (사업장수, 근로자수, 재해자수, 사망자수)
qty_cols = [col for col in accident_total.columns if any(x in col for x in ['사업장수', '근로자수', '재해자수', '사망자수'])]
accident_total_quantity = accident_total.melt(id_vars=union_column, value_vars=qty_cols, var_name='항목', value_name='수치')

# 그룹 2: 비율 및 위험도 (재해율, 사망만인율)
rate_cols = [col for col in accident_total.columns if any(x in col for x in ['재해율', '사망만인율'])]
accident_total_rate = accident_total.melt(id_vars=union_column, value_vars=rate_cols, var_name='항목', value_name='수치')

# 연도와 지표명 깔끔하게 분리 (예: 2019_사업장수 (개소) -> 2019 / 사업장수)
for df in [accident_total_quantity, accident_total_rate]:
    df[['연도', '지표']] = df['항목'].str.split('_', expand=True, n=1)
    df['지표'] = df['지표'].str.replace(r' \(.*\)', '', regex=True) # 단위 제거
    df.drop(columns=['항목'], inplace=True)

In [ ]:
accident_quantity = accident_quantity[['업종별 중분류','연도','지표','수치']]
accident_total_quantity = accident_total_quantity[['업종별 중분류','연도','지표','수치']]
accident_rate = accident_rate[['업종별 중분류','연도','지표','수치']]
accident_total_rate = accident_total_rate[['업종별 중분류','연도','지표','수치']]

## 저장_최종.csv

In [ ]:
accident_quantity.to_csv('data/accident_quantity.csv', index=False)
accident_rate.to_csv('data/accident_rate.csv', index=False)
accident_total_quantity.to_csv('data/accident_total_quantity.csv', index=False)
accident_total_rate.to_csv('data/accident_total_rate.csv', index=False)

# 산업 규모 및 임금별 근로시간

In [ ]:
payment_time

In [ ]:
payment_time_total

## 일단 헤더 들어내고...

In [ ]:
# 1. 0번 행(노동비용총액 등)을 새로운 컬럼 이름으로 지정
payment_time.columns = payment_time.iloc[0]
payment_time_total.columns = payment_time_total.iloc[0]

# 2. 이름으로 써먹은 0번 행은 이제 데이터에서 삭제
payment_time = payment_time.drop(0)
payment_time_total = payment_time_total.drop(0)

# 너도 나가!
payment_time.drop('규모별(1)', axis=1, inplace=True)
payment_time_total.drop('규모별(1)', axis=1, inplace=True)

In [ ]:
payment_time

In [ ]:
payment_time_total

## (1), (2)가 사라지는 마법!

In [ ]:
# (1), (2) 떼고
payment_time.columns = payment_time.columns.str.strip()
payment_time_total.columns = payment_time_total.columns.str.strip()

payment_time.rename(columns={'산업분류(2)':'산업분류'}, inplace=True)
payment_time_total.rename(columns={'산업분류(1)':'산업분류'}, inplace=True)

In [ ]:
payment_time.drop(columns='산업분류(1)', inplace=True) # 너도 나가

In [ ]:
payment_time

## 연도를 끼얹어주세용

In [ ]:
payment_time.columns

In [ ]:
# 연도향을 첨가해보아요
years = [ 2020, 2021, 2022, 2023, 2024]
base_names = ['전체근로일수 (일)', '상용근로일수 (일)', '임시일용근로일수 (일)', '전체근로시간 (시간)',
       '상용총근로시간 (시간)', '상용소정실근로시간 (시간)', '상용초과근로시간 (시간)', '임시일용근로시간 (시간)',
       '전체임금총액 (원)', '상용임금총액 (원)', '상용정액급여 (원)', '상용초과급여 (원)', '상용특별급여 (원)',
       '임시일용임금총액 (원)'] # total도 동일

final_cols = []
for year in years:
    for name in base_names:
        final_cols.append(f"{year}_{name}")

# 3. 맨 앞에 '기업규모별' 컬럼이 있다면 추가 (데이터프레임 구조에 맞게)
if len(payment_time.columns) == len(final_cols) + 1:
    final_cols = ['산업분류'] + final_cols

# 4. 이름표 갈아끼우기 (행은 전혀 건드리지 않음)
payment_time.columns = final_cols
payment_time_total.columns = final_cols

In [ ]:
payment_time

## 쓰읍 얘도 분리해야것소...
- 일/시간 함께 묶고 원끼리 묶겠습니다.

In [ ]:
union_column = ['산업분류']

# 그룹 1: 일, 시간
day_cols = [col for col in payment_time.columns if any(x in col for x in ['(일)','(시간)'])]
payment_time_date = payment_time.melt(id_vars=union_column, value_vars=day_cols, var_name='항목', value_name='수치')

# 그룹 2: 돈
money_cols = [col for col in payment_time.columns if any(x in col for x in ['(원)'])]
payment_time_money = payment_time.melt(id_vars=union_column, value_vars=money_cols, var_name='항목', value_name='비용')

# 연도와 지표명 깔끔하게 분리 (예: 2019_사업장수 (개소) -> 2019 / 사업장수)
for df in [payment_time_date, payment_time_money]:
    df[['연도', '지표']] = df['항목'].str.split('_', expand=True, n=1)
    df['지표'] = df['지표'].str.replace(r' \(.*\)', '', regex=True) # 단위 제거
    df.drop(columns=['항목'], inplace=True)

In [ ]:
payment_time_date = payment_time_date[['산업분류', '연도', '지표', '수치']]
payment_time_money = payment_time_money[['산업분류', '연도', '지표', '비용']]
payment_time_money

In [ ]:
# 똑같은거 한번 더 해주시면 됩니다.
union_column = ['산업분류']

# 그룹 1: 일, 시간
day_cols = [col for col in payment_time_total.columns if any(x in col for x in ['(일)','(시간)'])]
payment_time_total_date = payment_time_total.melt(id_vars=union_column, value_vars=day_cols, var_name='항목', value_name='수치')

# 그룹 2: 돈
money_cols = [col for col in payment_time_total.columns if any(x in col for x in ['(원)'])]
payment_time_total_money = payment_time_total.melt(id_vars=union_column, value_vars=money_cols, var_name='항목', value_name='비용')

# 연도와 지표명 깔끔하게 분리 (예: 2019_사업장수 (개소) -> 2019 / 사업장수)
for df in [payment_time_total_date, payment_time_total_money]:
    df[['연도', '지표']] = df['항목'].str.split('_', expand=True, n=1)
    df['지표'] = df['지표'].str.replace(r' \(.*\)', '', regex=True) # 단위 제거
    df.drop(columns=['항목'], inplace=True)

In [ ]:
payment_time_total_date = payment_time_total_date[['산업분류', '연도', '지표', '수치']]
payment_time_total_money = payment_time_total_money[['산업분류', '연도', '지표', '비용']]

## 알파벳 나가!

In [ ]:
payment_total_list = payment_time_total_money['산업분류'].tolist()
payment_total_list

In [ ]:
payment_total_list = [re.sub(re_pattern, '', col) for col in payment_total_list]
payment_total_list = [re.sub(no_braket, '', col) for col in payment_total_list]

payment_time_total_money['산업분류'] = payment_total_list

payment_time_total_money

In [ ]:
payment_total_list

In [ ]:
payment_total_list = payment_time_total_date['산업분류'].tolist()

payment_total_list = [re.sub(re_pattern, '', col) for col in payment_total_list]
payment_total_list = [re.sub(no_braket, '', col) for col in payment_total_list]

payment_time_total_date['산업분류'] = payment_total_list

payment_time_total_date

In [ ]:
payment_total_list = payment_time_date['산업분류'].tolist()

payment_total_list = [re.sub(re_pattern, '', col) for col in payment_total_list]
payment_total_list = [re.sub(no_braket, '', col) for col in payment_total_list]

payment_time_date['산업분류'] = payment_total_list

payment_time_date

In [ ]:
payment_total_list = payment_time_money['산업분류'].tolist()

payment_total_list = [re.sub(re_pattern, '', col) for col in payment_total_list]
payment_total_list = [re.sub(no_braket, '', col) for col in payment_total_list]

payment_time_money['산업분류'] = payment_total_list

payment_time_money

## 어디가 돈단위 바꿔야지
- 얘는 원입니다. 네.
- 근데 저거 썡으로 나누면 소수점 아레 네자기라 보기 싫잖아요? 그래서 반올림할겁니다... ~~구레나룻... 아니... 소수점 아래 두자리는 남겨주세요~~

In [ ]:
payment_time_money['비용'] = payment_time_money['비용'].astype(float)
payment_time_total_money['비용'] = payment_time_total_money['비용'].astype(float)

In [ ]:
payment_time_money['비용(만원)'] = round(payment_time_money['비용'] / 10000, 2)
payment_time_total_money['비용(만원)'] = round(payment_time_total_money['비용'] / 10000, 2)

## 내 하드에 저-장

In [ ]:
payment_time_date.to_csv('data/payment_time_date.csv', index=False)
payment_time_money.to_csv('payment_time_money.csv', index=False)
payment_time_total_date.to_csv('data/payment_time_total_date.csv', index=False)
payment_time_total_money.to_csv('data/payment_time_total_money.csv', index=False)

## 추가 가공: 위 니드 어 콩캣
- 이게 산업 분류따라서 나뉘었습니다... 네...
- 근데 이걸 합쳐도 되나? 싶으시죠? 어차피 제조업 전체 볼거라서 합쳤습니다.

### 예들아 이제 나가줄래?

In [ ]:
# 응 너 나가
payment_time_part1.drop('규모별(1)', axis=1, inplace=True)
payment_time_part2.drop('규모별(1)', axis=1, inplace=True)

In [ ]:
# 1. 0번 행(노동비용총액 등)을 새로운 컬럼 이름으로 지정
payment_time_part1.columns = payment_time_part1.iloc[0]
payment_time_part2.columns = payment_time_part2.iloc[0]

# 2. 이름으로 써먹은 0번 행은 이제 데이터에서 삭제
payment_time_part1 = payment_time_part1.drop(0)
payment_time_part2 = payment_time_part2.drop(0)

In [ ]:
payment_time_part1.drop('산업분류(1)', axis = 1, inplace=True)

### 응 괄호도 나가

In [ ]:
# (1), (2) 떼고
payment_time_part1.columns = payment_time_part1.columns.str.strip()
payment_time_part2.columns = payment_time_part2.columns.str.strip()

payment_time_part2.rename(columns={'산업분류별(1)':'산업분류'}, inplace=True)

In [ ]:
payment_time_part2

### 연도 추가 후 칼람명 변경

In [ ]:
# 연도향을 첨가해보아요
years = [ 2020, 2021, 2022, 2023, 2024]
base_names = ['전체근로일수 (일)', '상용근로일수 (일)', '임시일용근로일수 (일)', '전체근로시간 (시간)',
       '상용총근로시간 (시간)', '상용소정실근로시간 (시간)', '상용초과근로시간 (시간)', '임시일용근로시간 (시간)',
       '전체임금총액 (원)', '상용임금총액 (원)', '상용정액급여 (원)', '상용초과급여 (원)', '상용특별급여 (원)',
       '임시일용임금총액 (원)'] # total도 동일

final_cols = []
for year in years:
    for name in base_names:
        final_cols.append(f"{year}_{name}")

# 3. 맨 앞에 '기업규모별' 컬럼이 있다면 추가 (데이터프레임 구조에 맞게)
if len(payment_time.columns) == len(final_cols) + 1:
    final_cols = ['산업분류'] + final_cols

# 4. 이름표 갈아끼우기 (행은 전혀 건드리지 않음)
payment_time_part1.columns = final_cols

In [ ]:
payment_time_part1

In [ ]:
# 연도향을 첨가해보아요
years = [2015, 2016, 2017, 2018, 2019]
base_names = ['전체근로일수 (일)', '상용근로일수 (일)', '임시일용근로일수 (일)', '전체근로시간 (시간)',
       '상용총근로시간 (시간)', '상용소정실근로시간 (시간)', '상용초과근로시간 (시간)', '임시일용근로시간 (시간)',
       '전체임금총액 (원)', '상용임금총액 (원)', '상용정액급여 (원)', '상용초과급여 (원)', '상용특별급여 (원)',
       '임시일용임금총액 (원)'] # total도 동일

final_cols = []
for year in years:
    for name in base_names:
        final_cols.append(f"{year}_{name}")

# 3. 맨 앞에 '기업규모별' 컬럼이 있다면 추가 (데이터프레임 구조에 맞게)
if len(payment_time_part2.columns) == len(final_cols) + 1:
    final_cols = ['산업분류'] + final_cols

# 4. 이름표 갈아끼우기 (행은 전혀 건드리지 않음)
payment_time_part2.columns = final_cols

In [ ]:
payment_time_part2

In [ ]:
# '제조업' 단어가 포함된 모든 셀을 '제조업'으로 변경
payment_time_part2.loc[payment_time_part2['산업분류'].str.contains('제조업', na=False), '산업분류'] = '제조업'

In [ ]:
payment_time_part2

### 묶자...

In [ ]:
payment_time_all = pd.concat([payment_time_part2, payment_time_part1], axis=1)
payment_time_all

### 네 이제 멜트하고 단위 바꿔주시면 됩니다.

In [ ]:
union_column = ['산업분류']

# 그룹 1: 일, 시간
day_cols = [col for col in payment_time_all.columns if any(x in col for x in ['(일)','(시간)'])]
payment_time_all_date = payment_time_all.melt(id_vars=union_column, value_vars=day_cols, var_name='항목', value_name='수치')

# 그룹 2: 돈
money_cols = [col for col in payment_time_all.columns if any(x in col for x in ['(원)'])]
payment_time_all_money = payment_time_all.melt(id_vars=union_column, value_vars=money_cols, var_name='항목', value_name='비용')

# 연도와 지표명 깔끔하게 분리 (예: 2019_사업장수 (개소) -> 2019 / 사업장수)
for df in [payment_time_all_date, payment_time_all_money]:
    df[['연도', '지표']] = df['항목'].str.split('_', expand=True, n=1)
    df['지표'] = df['지표'].str.replace(r' \(.*\)', '', regex=True) # 단위 제거
    df.drop(columns=['항목'], inplace=True)

In [ ]:
payment_time_all_date

#### 왜 맨날 순서가 뻑나는것이며

In [ ]:
payment_time_all_date = payment_time_all_date[['산업분류', '연도', '지표', '수치']]
payment_time_all_money = payment_time_all_money[['산업분류', '연도', '지표', '비용']]

In [ ]:
payment_time_all_money

### 돈단위 (만원)으로 변경

In [ ]:
payment_time_all_money['비용'] = payment_time_all_money['비용'].astype(float)
payment_time_all_money['비용(만원)'] = round(payment_time_all_money['비용'] / 10000, 2)

### 내 하드에 저장

In [ ]:
payment_time_all_date.to_csv('data/payment_time_all_date.csv', index=False)
payment_time_all_money.to_csv('data/payment_time_all_money.csv', index=False)

# 손익계산서
- 저 사실 재무제표 어느정도는 볼 줄 압니다. 네.
- 행에는 안 쓰여있지만 단위가 (백만원)입니다. 

In [ ]:
sonic

In [ ]:
sonic_jp

In [ ]:
sonic_jejo

In [ ]:
sonic_jp_jejo

## 저기 기업규모 날리고 가실게요~

In [ ]:
# 날리고 멜팅합시다.
df_list = [sonic, sonic_jp, sonic_jejo, sonic_jp_jejo]

for df in df_list:
    df.drop('기업규모별', axis=1, inplace=True)

In [ ]:
sonic

- 이 얼탱이없는 반복문은 왜 나왔느냐... 간단합니다. 똑같은거 네 개 할거면 걍 반복문 돌려도 되지 않음? 해서 나온겁니다.

## 업종코드 제거

In [ ]:
sonic['업종코드별'].tolist()
# 알파벳이 한글자 아니면 세글자네요.

In [ ]:
no_code = r'[A-Z,0-9, -]+' # 규식정

for df in df_list:
    df_col = df['업종코드별'].tolist()
    df_col = [re.sub(no_code, '', col) for col in df_col]
    df['업종코드별'] = df_col # 작용_최종.py


In [ ]:
sonic

## 멜트다운!
- 근데 반복문 돌릴거라 저장도 같이 되는...

In [ ]:
# 파이참은 셀이 100개가 넘어가면 뻗는군요...
# 멜트다운을 어떻게 할거냐면 업종-년도-계정과목-액수로 할거예요. 감 좀 오시져?
# 그렇게 해야 우리가 그룹바이 하기가 편합니다. 대신 얘는 칼럼갖고 노가다는 안 해도 되니 다행이군요.
filename_list = ['data/sonic.csv', 'data/sonic_jp.csv', 'data/sonic_jejo.csv', 'data/sonic_jp_jejo.csv']
# jp = Job Paymemt의 약자(재팬 아님)

for i, df in enumerate(df_list):
    df_melted = df.melt(id_vars=['업종코드별', '계정항목별'], var_name='연도', value_name='비용')
    df_melted['비용'] = pd.to_numeric(df_melted['비용'], errors='coerce') # 인자 너는 숫자다잉
    df_melted['연도'] = df_melted['연도'].astype(int) # 응 너도
    df_melted.to_csv(filename_list[i], index=False)